# Phase 11 — Deployment and Prediction Interface

**Project:** Design a Machine Learning Approach to Analyse Students’ Performance Based on their Socio-economic Status in the Kingdom of Bahrain  
**Purpose:** Package the governed Phase 8 model, validate deployment readiness, generate a privacy-preserving inference engine, and create a bilingual Streamlit decision-support interface.

> Predictions are decision-support outputs only. They must not be used for automatic punishment, exclusion, labelling, or denial of educational opportunities.

## Section 11.1 — Environment and Deployment Configuration

This section sets the deployment mode and locates the Phase 8 and Phase 10 governance artifacts. `PRODUCTION` is permitted only when all final research gates have passed.

In [1]:
from pathlib import Path
import json, shutil, sys, os, hashlib, zipfile
from datetime import datetime
import joblib
import numpy as np
import pandas as pd

BASE_DIR = Path('/mnt/data') if Path('/mnt/data').exists() else Path.cwd()
PHASE_11_DIR = BASE_DIR / 'Phase_11_Deployment_Package'
PHASE_11_DIR.mkdir(parents=True, exist_ok=True)

REQUESTED_DEPLOYMENT_MODE = 'AUTO'  # AUTO, DEVELOPMENT_DEMO, or PRODUCTION
MAX_BATCH_ROWS = 5000
DO_NOT_LOG_INPUTS = True

print('Base directory:', BASE_DIR)
print('Requested mode:', REQUESTED_DEPLOYMENT_MODE)

Base directory: C:\Users\User\Desktop\All
Requested mode: AUTO


## Section 11.2 — Governance Gate and Artifact Discovery

In [2]:
def load_json(path):
    with Path(path).open('r', encoding='utf-8') as handle:
        return json.load(handle)

def locate_one(patterns):
    candidates = []
    for pattern in patterns:
        candidates.extend(BASE_DIR.rglob(pattern))
    candidates = [p for p in candidates if p.is_file()]
    if not candidates:
        raise FileNotFoundError(f'No file found for patterns: {patterns}')
    return max(candidates, key=lambda p: p.stat().st_mtime)

PHASE_10_HANDOFF = locate_one(['phase_10_handoff_manifest_for_phase_11.json'])
PHASE_8_COMPLETION = locate_one(['phase_08_completion_manifest*.json'])
PHASE_8_BUNDLE = locate_one(['phase_08_final_model_bundle*.joblib'])
PHASE_8_MODEL_CARD = locate_one(['phase_08_final_model_card*.json'])
TRAINING_DATASET = locate_one(['phase_03_training_dataset_for_phase_04.csv'])

phase10 = load_json(PHASE_10_HANDOFF)
phase8 = load_json(PHASE_8_COMPLETION)
model_card = load_json(PHASE_8_MODEL_CARD)

production_ready = bool(
    phase10.get('final_report_ready')
    and phase10.get('phase_4_deep_tuning_frozen')
    and phase10.get('phase_8_final_holdout_completed')
    and phase10.get('phase_9_topics_manually_reviewed')
    and phase10.get('final_model_bundle')
)

if REQUESTED_DEPLOYMENT_MODE == 'AUTO':
    DEPLOYMENT_MODE = 'PRODUCTION' if production_ready else 'DEVELOPMENT_DEMO'
elif REQUESTED_DEPLOYMENT_MODE == 'PRODUCTION' and not production_ready:
    raise RuntimeError('PRODUCTION was requested, but the Phase 10 final governance gates have not passed.')
else:
    DEPLOYMENT_MODE = REQUESTED_DEPLOYMENT_MODE

print('Deployment mode:', DEPLOYMENT_MODE)
print('Production ready:', production_ready)
print('Phase 8 final evaluation completed:', phase8.get('final_evaluation_completed'))

Deployment mode: DEVELOPMENT_DEMO
Production ready: False
Phase 8 final evaluation completed: False


## Section 11.3 — Load and Validate the Final Pipeline

In [3]:
bundle = joblib.load(PHASE_8_BUNDLE)
pipeline = bundle['artifact']
predictor_columns = list(bundle['predictor_columns'])
class_order = list(bundle['class_order'])
target_mapping = dict(bundle['target_mapping'])

required_bundle_keys = {
    'artifact', 'model_name', 'model_stage', 'predictor_columns',
    'target_mapping', 'class_order', 'training_dataset_sha256'
}
missing_keys = sorted(required_bundle_keys - set(bundle))
if missing_keys:
    raise KeyError(f'Model bundle is missing keys: {missing_keys}')
if not hasattr(pipeline, 'predict') or not hasattr(pipeline, 'predict_proba'):
    raise TypeError('The final artifact must support predict and predict_proba.')

print('Model:', bundle['model_name'])
print('Stage:', bundle['model_stage'])
print('Predictors:', len(predictor_columns))
print('Classes:', class_order)

Model: Logistic Regression
Stage: Baseline
Predictors: 17
Classes: ['Low', 'Medium', 'High']


## Section 11.4 — Build the Governed Input Schema

In [4]:
training_data = pd.read_csv(TRAINING_DATASET)
missing_predictors = [c for c in predictor_columns if c not in training_data.columns]
if missing_predictors:
    raise KeyError(f'Training dataset is missing predictors: {missing_predictors}')

# The schema is built from the official training set only. The raw training set is not copied into deployment.
feature_schema = {}
for column in predictor_columns:
    series = training_data[column]
    if pd.api.types.is_numeric_dtype(series):
        feature_schema[column] = {
            'type': 'integer' if pd.api.types.is_integer_dtype(series) else 'number',
            'minimum': float(series.min()),
            'maximum': float(series.max()),
            'default': float(series.median()),
        }
    else:
        options = [str(v) for v in series.dropna().value_counts().index.tolist()]
        feature_schema[column] = {
            'type': 'categorical',
            'options': options,
            'default': options[0],
        }

print(pd.DataFrame(feature_schema).T[['type', 'default']])

                                 type             default
father_alive              categorical                 Yes
mother_alive              categorical                 Yes
father_education          categorical          Bachelor's
mother_education          categorical          Bachelor's
father_job                categorical            Employed
mother_job                categorical          Unemployed
marital_status            categorical             Married
family_income             categorical  More than 1000 BHD
number_of_children            integer                 4.0
gender                    categorical                Male
school_type               categorical          Government
stage                     categorical             Primary
overall_grade_level           integer                 6.0
tutoring_support          categorical                  No
social_activities         categorical                 Yes
chronic_disease           categorical                  No
daily_smart_de

## Section 11.5 — Governed Prediction Engine

In [5]:
# The reusable implementation is written to inference_engine.py in the deployment package.
import importlib.util
engine_path = PHASE_11_DIR / 'inference_engine.py'
if not engine_path.exists():
    raise FileNotFoundError('Run the package-generation cells or use the delivered Phase 11 package.')

spec = importlib.util.spec_from_file_location('phase11_engine', engine_path)
module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(module)
engine = module.PredictionEngine(PHASE_11_DIR)
print('Prediction engine loaded successfully.')

FileNotFoundError: Run the package-generation cells or use the delivered Phase 11 package.

## Section 11.6 — Single-Record Prediction and Uncertainty

In [ ]:
default_record = {
    name: engine.schema['features'][name]['default']
    for name in engine.predictor_columns
}
result = engine.predict_one(default_record)
print(json.dumps(result, indent=2, ensure_ascii=False))

assert result['predicted_class'] in engine.class_order
assert abs(sum(result['probabilities'].values()) - 1.0) < 1e-9

## Section 11.7 — Model-Agnostic Local Sensitivity

In [ ]:
sensitivity = engine.local_sensitivity(default_record, max_features=10)
display(sensitivity)

print('Interpretation rule: the table measures one-variable-at-a-time probability sensitivity.')
print('It is not a causal explanation and must not be presented as proof that a factor causes performance.')

## Section 11.8 — Batch Prediction Pipeline

In [ ]:
batch_example = pd.DataFrame([default_record, default_record])
batch_predictions = engine.predict_batch(batch_example)
display(batch_predictions)

assert len(batch_predictions) == len(batch_example)
assert (batch_predictions['prediction_status'] == 'success').all()

## Section 11.9 — Streamlit Interface Package

In [ ]:
required_files = [
    'app.py', 'inference_engine.py', 'requirements.txt', 'README.txt',
    'run_app_windows.bat', 'run_app_linux_mac.sh', 'Dockerfile',
    'config/deployment_schema.json', 'config/deployment_manifest.json',
    'config/model_card.json', 'models/final_model_bundle.joblib',
    'templates/empty_batch_template.csv', 'templates/sample_batch_input.csv',
]
missing = [name for name in required_files if not (PHASE_11_DIR / name).exists()]
if missing:
    raise FileNotFoundError(f'Deployment package is incomplete: {missing}')
print('Streamlit package files are complete.')

## Section 11.10 — Privacy, Safety, and Quality Checks

In [ ]:
quality_path = PHASE_11_DIR / 'reports' / 'phase_11_quality_checks.csv'
quality_checks = pd.read_csv(quality_path)
display(quality_checks)
if not quality_checks['passed'].astype(bool).all():
    failed = quality_checks.loc[~quality_checks['passed'].astype(bool), 'quality_check'].tolist()
    raise AssertionError(f'Phase 11 quality checks failed: {failed}')

packaged_names = [str(p.relative_to(PHASE_11_DIR)).lower() for p in PHASE_11_DIR.rglob('*') if p.is_file()]
assert not any('holdout' in name for name in packaged_names)
assert not any('training_dataset' in name for name in packaged_names)
print('Privacy and package-content checks passed.')

## Section 11.11 — Deployment Readiness and Final Export

In [ ]:
readiness = pd.read_csv(PHASE_11_DIR / 'reports' / 'phase_11_deployment_readiness.csv')
display(readiness)

if DEPLOYMENT_MODE == 'PRODUCTION':
    assert readiness.loc[readiness['required_for_production'], 'passed'].astype(bool).all()
    print('PRODUCTION deployment gates passed.')
else:
    print('Development demo generated. Final student decisions are prohibited.')

## Running the Interface

### Windows
Double-click `run_app_windows.bat` inside the extracted deployment package.

### Command line
```bash
python -m pip install -r requirements.txt
python -m streamlit run app.py
```

The application opens locally at `http://localhost:8501`.